# Sprint 53 · Task B · Step B1 — baseline + data split

Runtime: **CPU**

| Cell | What it does |
|---|---|
| 1 | Upload `sprint53_taskA_final.zip`, then `sprint53_taskB_step1_code.zip` (one at a time) |
| 2 | Build the development / held-out split |
| 3 | Validate the split and compute the baseline numbers per half |
| 4 | Download a checkpoint zip (keep it — Step B2/B3 start from it) |

In [1]:
# 1. Upload and unpack (asks one file at a time until both are present)
import os, zipfile, glob, shutil
from google.colab import files
os.chdir('/content')

def unpack(path):
    with zipfile.ZipFile(path) as z:
        tops = {p.split('/')[0] for p in z.namelist() if p.strip('/')}
        if tops != {'sprint53'}:
            raise SystemExit(f'!! {path} does not contain a single sprint53/ folder (found {tops}).')
        z.extractall('/content')
    print('   unpacked', path)

have_data = lambda: os.path.exists('/content/sprint53/results/P11_E10_rationale.jsonl')
have_code = lambda: os.path.exists('/content/sprint53/taskB/make_split.py')
code_zips = []
while not (have_data() and have_code()):
    need = []
    if not have_data(): need.append('Task A project (sprint53_taskA_final.zip)')
    if not have_code(): need.append('Task B Step 1 code (sprint53_taskB_step1_code.zip)')
    print('Please upload ONE file now:', ' / '.join(need))
    for name in files.upload():
        unpack(name)
        if 'taskb' in name.lower(): code_zips.append(name)
        else:
            for c in code_zips: unpack(c)   # newer code always wins

os.chdir('/content/sprint53')
shutil.rmtree('data/taskB', ignore_errors=True)   # always rebuild the split here, from scratch
print('\nReady. Task B code:', sorted(os.listdir('taskB')))

Please upload ONE file now: Task A project (sprint53_taskA_final.zip) / Task B Step 1 code (sprint53_taskB_step1_code.zip)


Saving sprint53_taskA_final.zip to sprint53_taskA_final.zip
   unpacked sprint53_taskA_final.zip
Please upload ONE file now: Task B Step 1 code (sprint53_taskB_step1_code.zip)


Saving sprint53_taskB_step1_code.zip to sprint53_taskB_step1_code.zip
   unpacked sprint53_taskB_step1_code.zip

Ready. Task B code: ['BASELINE_AND_METRICS.md', 'make_split.py', 'validate_split.py']


In [2]:
# 2. Build the split  (expected: 'Chosen seed 1324 (penalty 3.20). dev sources 20, held-out sources 20; repeat subset 40 items')
!python taskB/make_split.py
!wc -l data/taskB/*.jsonl

Chosen seed 1324 (penalty 3.20). dev sources 20, held-out sources 20; repeat subset 40 items -> data/taskB/
    28 data/taskB/dev_pairs.jsonl
    20 data/taskB/dev_setA.jsonl
   138 data/taskB/dev_setB.jsonl
    32 data/taskB/heldout_pairs.jsonl
    20 data/taskB/heldout_setA.jsonl
   137 data/taskB/heldout_setB.jsonl
    40 data/taskB/repeat_subset.jsonl
   415 total


In [3]:
# 3. Validate + baseline reference  (must end the checks with 'HARD FAILURES: 0' and 'WARNINGS: 0')
!python taskB/validate_split.py

TASK B · STEP B1 — SPLIT VALIDATION AND BASELINE REFERENCE
Split seed 1324 (best of 2000 candidates, penalty 3.2)

DEV      sources 20  Set B rows 138  pairs 28  Set A 20 (human mean 84.0, sd 9.8, clean 9, moderate 11)
         categories {'ambiguity_context': 3, 'culture': 2, 'idiom': 3, 'named_entity': 3, 'numbers_units_dates': 2, 'register': 3, 'simple_control': 2, 'terminology': 2}
         variants   {'fluent_wrong': 20, 'gold': 20, 'literal': 20, 'original_candidate': 13, 'padded': 3, 'paraphrase': 20, 'targeted_error': 20, 'traditional_char': 15, 'traditional_tw': 7}
HELDOUT  sources 20  Set B rows 137  pairs 32  Set A 20 (human mean 84.0, sd 11.0, clean 10, moderate 10)
         categories {'ambiguity_context': 2, 'culture': 3, 'idiom': 2, 'named_entity': 3, 'numbers_units_dates': 3, 'register': 2, 'simple_control': 2, 'terminology': 3}
         variants   {'fluent_wrong': 20, 'gold': 20, 'literal': 20, 'original_candidate': 13, 'padded': 2, 'paraphrase': 20, 'targeted_error': 

In [4]:
# 4. Download a checkpoint of the whole project (data, results, Task A + Task B code)
import datetime, zipfile, os
from google.colab import files
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M')
out = f'/content/sprint53_taskB_B1_{stamp}.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, fnames in os.walk('/content/sprint53'):
        dirs[:] = [d for d in dirs if d not in ('node_modules', '.venv', '__pycache__')]
        for f in fnames:
            if f.endswith('.zip'): continue
            full = os.path.join(root, f); z.write(full, os.path.relpath(full, '/content'))
print(f'{out} ({os.path.getsize(out)/1e6:.1f} MB) - downloading...')
files.download(out)

/content/sprint53_taskB_B1_20261003_0134.zip (2.0 MB) - downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>